# Track Parameter Pull Distribution Analysis

This notebook performs a validation analysis of track reconstruction using pull distributions for the Kalman Filter and seeding algorithms. It compares reconstructed track parameters against Monte Carlo truth values to assess the quality of the track parameter estimation and their uncertainties.

## Overview

The analysis evaluates **pull distributions** for five track parameters at two stages:
- **Seed level**: Initial track parameter estimates
- **Kalman Filter (KF) level**: Updated track parameters after filtering

## Track Parameters Analyzed

The five track parameters examined are:
1. **s₀ = y**: Transverse position
2. **s₁ = z**: Longitudinal position  
3. **s₂ = sin φ**: Azimuthal angle
4. **s₃ = tan λ**: Dip angle
5. **s₄ = q/pₜ**: Signed inverse transverse momentum

## Pull Distribution Definition

For each parameter, the pull is calculated as:

**Pull = (reconstructed - true) / √(covariance)**

Ideally, pull distributions should be Gaussian with mean = 0 and σ = 1, indicating:
- Unbiased parameter estimation (mean = 0)
- Correct uncertainty estimation (σ = 1)

## Outputs

The notebook generates plots showing pull distributions fitted with Gaussian functions, saved as `.eps` and `.png` files in the `Units/` directory for both seed and Kalman filter stages.

In [ ]:
### Python script to load the fastSimulation library and set up the plotting environment
import os
import ROOT
from Plot_func import SetHisto
from Plot_func import SetEff
from Plot_func import SetCanvas
from Plot_func import SetLegend
from Plot_func import SetGlobalStyle
from Plot_func import SetColor
from ROOT import  gROOT, gSystem
gSystem.Load("../aliKalman/AliExternalTrackParam.so")
gROOT.LoadMacro("../MC/fastSimulation.cxx+")
gROOT.LoadMacro("../MC/fastSimulationTest.C")

In [ ]:
### Read input data
folder = "../data/"

foldercheck="Units/"
os.makedirs(foldercheck, exist_ok=True)
inputData = folder+"fastParticle.list"
tree  = ROOT.AliXRDPROOFtoolkit.MakeChainRandom(inputData,"fastPart",chr(0),10000)

In [ ]:
#### Function to plot unit test histograms with Gaussian fit
def UnitTot(tree1,var,varname,ch0,legend0,ln,lv,f,histopar,histoheight):
    SetColor()
    SetGlobalStyle()

    f.SetParameters(10000,0,1)
    f.SetLineWidth(3)
    tree1.Draw(var+">>h0("+histopar+")","isOK")
    histo0 = ROOT.gPad.GetPrimitive("h0")

    SetHisto(histo0,";"+varname+";Entries (a.u.)",ROOT.kBlue,20,[0,histoheight])
    histo0.GetXaxis().SetTitleSize(0.065)
    histo0.GetXaxis().SetTitleOffset(0.95)
    histo0.GetYaxis().SetTitleSize(0.065)
    histo0.GetYaxis().SetTitleOffset(1.2)
    histo0.Fit("fgaus")
    SetCanvas(ch0)
    histo0.Draw("E0")
    SetLegend(legend0)
    legend0.SetTextSize(0.05*0.68*1.6)
    legend0.SetHeader(lv)
    legend0.AddEntry(histo0,ln,"pl")
    legend0.AddEntry(f," Gauss fit :","l")
    legend0.AddEntry(0," #mu = "+"%0.3f"%f.GetParameter(1)+" #pm ""%0.3f"% f.GetParError(1),"")
    legend0.AddEntry(0," #sigma = "+"%0.3f"%f.GetParameter(2)+" #pm ""%0.3f"% f.GetParError(2),"")

## Results
The resulting pull distribution plots are saved in the `Units/` directory as `.eps` and `.png` files, providing visual validation of the track parameter estimation quality at both seed and Kalman filter levels.

### Seed

In [ ]:
tree.SetAlias("isOK","part.fParamIn[1].fP[4]!=0 && part.fParamIn@.size()>50")

tree.SetAlias("p0MCS","part.fParamMC[part.fFirstIndexIn-1].fP[0]")
tree.SetAlias("p0InS","part.fParamIn[part.fFirstIndexIn-1].fP[0]")
tree.SetAlias("p1MCS","part.fParamMC[part.fFirstIndexIn-1].fP[1]")
tree.SetAlias("p1InS","part.fParamIn[part.fFirstIndexIn-1].fP[1]")
tree.SetAlias("p2MCS","part.fParamMC[part.fFirstIndexIn-1].fP[2]")
tree.SetAlias("p2InS","part.fParamIn[part.fFirstIndexIn-1].fP[2]")
tree.SetAlias("p3MCS","part.fParamMC[part.fFirstIndexIn-1].fP[3]")
tree.SetAlias("p3InS","part.fParamIn[part.fFirstIndexIn-1].fP[3]")
tree.SetAlias("p4MCS","part.fParamMC[part.fFirstIndexIn-1].fP[4]")
tree.SetAlias("p4InS","part.fParamIn[part.fFirstIndexIn-1].fP[4]")
tree.SetAlias("pMCS","part.fParamMC[part.fFirstIndexIn-1].P()")
tree.SetAlias("pInS","part.fParamIn[part.fFirstIndexIn-1].P()")

tree.SetAlias("Unit0Seed","(p0InS-p0MCS)/sqrt(part.fParamIn[part.fFirstIndexIn-1].fC[0]/2)")
tree.SetAlias("Unit1Seed","(p1InS-p1MCS)/sqrt(part.fParamIn[part.fFirstIndexIn-1].fC[2]/2)")
tree.SetAlias("Unit2Seed","(p2InS-p2MCS)/sqrt(part.fParamIn[part.fFirstIndexIn-1].fC[5]/2)")
tree.SetAlias("Unit3Seed","(p3InS-p3MCS)/sqrt(part.fParamIn[part.fFirstIndexIn-1].fC[9]/2)")
tree.SetAlias("Unit4Seed","(p4InS-p4MCS)/sqrt(part.fParamIn[part.fFirstIndexIn-1].fC[14]/2)")

In [ ]:
var = "Unit0Seed"
varname = "( #it{s}_{0}^{true} - #it{s}_{0}^{reco} )/ #sqrt{#it{C}_{00}}"
lname = "  Seed Pull"
lvarname = "(a)  #it{s_{0}} = #it{y}"
fgaus = ROOT.TF1("fgaus", "[0]*TMath::Gaus(x,[1],[2])", -4, 4)
c0 = ROOT.TCanvas("c0","c0",500,600)
l0 = ROOT.TLegend(0.35,0.64,0.96,0.91)
height = 150
range = "40,-4,4"
UnitTot(tree,var,varname,c0,l0,lname,lvarname,fgaus,range,height)
l0.Draw()
c0.Draw()
c0.Print(foldercheck+"Unit0Seed.eps")
c0.Print(foldercheck+"Unit0Seed.png")

In [ ]:
var = "Unit1Seed"
varname = "( #it{s}_{1}^{true} - #it{s}_{1}^{reco} )/ #sqrt{#it{C}_{11}}"
lname = "  Seed Pull"
lvarname = "(b)  #it{s_{1}} = #it{z}"
fgaus = ROOT.TF1("fgaus", "[0]*TMath::Gaus(x,[1],[2])", -10, 10)
fgaus.SetParLimits(2,0.2,20)
c0 = ROOT.TCanvas("c0","c0",500,600)
l0 = ROOT.TLegend(0.35,0.64,0.96,0.91)
height = 150
range = "40,-4,4"
UnitTot(tree,var,varname,c0,l0,lname,lvarname,fgaus,range,height)
l0.Draw()
c0.Draw()
c0.Print(foldercheck+"Unit1Seed.eps")
c0.Print(foldercheck+"Unit1Seed.png")

In [ ]:
var = "Unit2Seed"
varname = "( #it{s}_{2}^{true} - #it{s}_{2}^{reco} )/ #sqrt{#it{C}_{22}}"
lname = "  Seed Pull"
lvarname = "(\\text{c}) \ {s}_{2} = \sin{\phi}"
fgaus = ROOT.TF1("fgaus", "[0]*TMath::Gaus(x,[1],[2])", -10, 10)
c0 = ROOT.TCanvas("c0","c0",500,600)
l0 = ROOT.TLegend(0.35,0.64,0.96,0.91)
height = 150
range = "40,-4,4"
UnitTot(tree,var,varname,c0,l0,lname,lvarname,fgaus,range,height)
l0.Draw()
c0.Draw()
c0.Print(foldercheck+"Unit2Seed.eps")
c0.Print(foldercheck+"Unit2Seed.png")

In [ ]:
var = "Unit3Seed"
varname = "( #it{s}_{3}^{true} - #it{s}_{3}^{reco} )/ #sqrt{#it{C}_{33}}"
lname = "  Seed Pull"
lvarname = "(\\text{d})\ s_{3} = \\tan{\lambda}"
fgaus = ROOT.TF1("fgaus", "[0]*TMath::Gaus(x,[1],[2])", -4, 4)
c0 = ROOT.TCanvas("c0","c0",500,600)
l0 = ROOT.TLegend(0.35,0.64,0.96,0.91)
height = 150
range = "40,-4,4"
UnitTot(tree,var,varname,c0,l0,lname,lvarname,fgaus,range,height)
l0.Draw()
c0.Draw()
c0.Print(foldercheck+"Unit3Seed.eps")
c0.Print(foldercheck+"Unit3Seed.png")

In [ ]:
var = "Unit4Seed"
varname = "( #it{s}_{4}^{true} - #it{s}_{4}^{reco} )/ #sqrt{#it{C}_{44}}"
lname = "  CKF Pull"
lvarname = "(e)   #it{s}_{4} = #it{q/p}_{T} "
fgaus = ROOT.TF1("fgaus", "[0]*TMath::Gaus(x,[1],[2])", -10, 10)
c0 = ROOT.TCanvas("c0","c0",500,600)
l0 = ROOT.TLegend(0.35,0.64,0.96,0.91)
height = 150
range = "40,-4,4"
UnitTot(tree,var,varname,c0,l0,lname,lvarname,fgaus,range,height)
l0.Draw()
c0.Draw()
c0.Print(foldercheck+"Unit4Seed.eps")
c0.Print(foldercheck+"Unit4Seed.png")

### KF

In [ ]:
tree.SetAlias("isOK","part.fParamIn[1].fP[4]!=0 && part.fParamIn@.size()>50")

tree.SetAlias("p0MC","part.fParamMC[1].fP[0]")
tree.SetAlias("p0In","part.fParamIn[1].fP[0]")
tree.SetAlias("p1MC","part.fParamMC[1].fP[1]")
tree.SetAlias("p1In","part.fParamIn[1].fP[1]")
tree.SetAlias("p2MC","part.fParamMC[1].fP[2]")
tree.SetAlias("p2In","part.fParamIn[1].fP[2]")
tree.SetAlias("p3MC","part.fParamMC[1].fP[3]")
tree.SetAlias("p3In","part.fParamIn[1].fP[3]")
tree.SetAlias("p4MC","part.fParamMC[1].fP[4]")
tree.SetAlias("p4In","part.fParamIn[1].fP[4]")
tree.SetAlias("pMC","part.fParamMC[1].P()")
tree.SetAlias("pIn","part.fParamIn[1].P()")

tree.SetAlias("Unit0MC","(p0In-p0MC)/sqrt(part.fParamIn[1].fC[0])")
tree.SetAlias("Unit1MC","(p1In-p1MC)/sqrt(part.fParamIn[1].fC[2])")
tree.SetAlias("Unit2MC","(p2In-p2MC)/sqrt(part.fParamIn[1].fC[5])")
tree.SetAlias("Unit3MC","(p3In-p3MC)/sqrt(part.fParamIn[1].fC[9])")
tree.SetAlias("Unit4MC","(p4In-p4MC)/sqrt(part.fParamIn[1].fC[14])")


In [ ]:
var = "Unit0MC"
varname = "( #it{s}_{0}^{true} - #it{s}_{0}^{reco} )/ #sqrt{#it{C}_{00}}"
lname = "  CKF Pull"
lvarname = "(a)  #it{s_{0}} = #it{y}"
fgaus = ROOT.TF1("fgaus", "[0]*TMath::Gaus(x,[1],[2])", -10, 10)
c0 = ROOT.TCanvas("c0","c0",500,600)
l0 = ROOT.TLegend(0.35,0.64,0.96,0.91)
height = 150
range = "40,-4,4"
UnitTot(tree,var,varname,c0,l0,lname,lvarname,fgaus,range,height)
l0.Draw()
c0.Draw()
c0.Print(foldercheck+"Unit0.eps")
c0.Print(foldercheck+"Unit0.png")

In [ ]:
var = "Unit1MC"
varname = "( #it{s}_{1}^{true} - #it{s}_{1}^{reco} )/ #sqrt{#it{C}_{11}}"
lname = "  CKF Pull"
lvarname = "(b)  #it{s_{1}} = #it{z}"
fgaus = ROOT.TF1("fgaus", "[0]*TMath::Gaus(x,[1],[2])", -10, 10)
fgaus.SetParLimits(2,0.1,20)
c0 = ROOT.TCanvas("c0","c0",500,600)
l0 = ROOT.TLegend(0.35,0.64,0.96,0.91)
height = 150
range = "40,-4,4"
UnitTot(tree,var,varname,c0,l0,lname,lvarname,fgaus,range,height)
l0.Draw()
c0.Draw()
c0.Print(foldercheck+"Unit1.eps")
c0.Print(foldercheck+"Unit1.png")

In [ ]:
var = "Unit2MC"
varname = "( #it{s}_{2}^{true} - #it{s}_{2}^{reco} )/ #sqrt{#it{C}_{22}}"
lname = "  CKF Pull"
lvarname = "(\\text{c}) \ {s}_{2} = \sin{\phi}"
fgaus = ROOT.TF1("fgaus", "[0]*TMath::Gaus(x,[1],[2])", -10, 10)
c0 = ROOT.TCanvas("c0","c0",500,600)
l0 = ROOT.TLegend(0.35,0.64,0.96,0.91)
height = 150
range = "40,-4,4"
UnitTot(tree,var,varname,c0,l0,lname,lvarname,fgaus,range,height)
l0.Draw()
c0.Draw()
c0.Print(foldercheck+"Unit2.eps")
c0.Print(foldercheck+"Unit2.png")

In [ ]:
var = "Unit3MC"
varname = "( #it{s}_{3}^{true} - #it{s}_{3}^{reco} )/ #sqrt{#it{C}_{33}}"
lname = "  CKF Pull"
lvarname = "(\\text{d})\ s_{3} = \\tan{\lambda}"
fgaus = ROOT.TF1("fgaus", "[0]*TMath::Gaus(x,[1],[2])", -4, 4)
c0 = ROOT.TCanvas("c0","c0",500,600)
l0 = ROOT.TLegend(0.35,0.64,0.96,0.91)
height = 150
range = "40,-4,4"
UnitTot(tree,var,varname,c0,l0,lname,lvarname,fgaus,"40,-4,4",height)
l0.Draw()
c0.Draw()
c0.Print(foldercheck+"Unit3.eps")
c0.Print(foldercheck+"Unit3.png")

In [ ]:
var = "Unit4MC"
varname = "( #it{s}_{4}^{true} - #it{s}_{4}^{reco} )/ #sqrt{#it{C}_{44}}"
lname = "  CKF Pull"
lvarname = "(e)   #it{s}_{4} = #it{q/p}_{T} "
fgaus = ROOT.TF1("fgaus", "[0]*TMath::Gaus(x,[1],[2])", -10, 10)
c0 = ROOT.TCanvas("c0","c0",500,600)
l0 = ROOT.TLegend(0.35,0.64,0.96,0.91)
height = 150
range = "40,-4,4"
UnitTot(tree,var,varname,c0,l0,lname,lvarname,fgaus,range,height)
l0.Draw()
c0.Draw()
c0.Print(foldercheck+"Unit4.eps")
c0.Print(foldercheck+"Unit4.png")